# AEM data and descriptors

Inspect conductivity data and use source functions to calculate chemical descriptors and valid charged mutations.

Install the package from the repository root with `python -m pip install -e ".[research,notebooks]"`. All algorithms are imported from `src/latent_materials/`.

In [ ]:
from pathlib import Path
import pandas as pd
from latent_materials.cli import run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA = ROOT / "data"
RESULTS = ROOT / "results"


Load the supplied conductivity dataset.

In [ ]:
data = pd.read_csv(DATA / "aems/CVAE/data/modified_data_final.csv")
data.head()

Evaluate a charged example. The IEC value is a molecular proxy; the accessibility value is an in-house heuristic.

In [ ]:
from rdkit import Chem
from latent_materials.aems.descriptors import formal_charge, cationic_site_count, ion_exchange_capacity_proxy, sa_score_simple
molecule = Chem.MolFromSmiles("CCCC[N+](C)(C)C")
pd.Series({"formal_charge": formal_charge(molecule), "cationic_sites": cationic_site_count(molecule), "IEC_proxy_mmol_g": ion_exchange_capacity_proxy(molecule), "accessibility_heuristic": sa_score_simple(molecule)})

The same tested mutation operators are imported by the production generation functions.

In [ ]:
from latent_materials.aems.mutations import add_quatN, add_phosphonium, add_imidazolium, valid_canon, structure_pass
products = [operator("CCCC") for operator in [add_quatN, add_phosphonium, add_imidazolium]]
pd.DataFrame({"product": products, "canonical": [valid_canon(s) for s in products], "BO_gate_pass": [structure_pass(s) for s in products]})